# GLMsingle QC — Group Overview

Loops over all subjects, saves per-subject QC figures, and produces a group-level summary.

- Per-subject figures → `<glmsingle_dir>/sub-<id>/figures/qc_<section>.png`
- Group summary CSV → `<glmsingle_dir>/qc_group_summary.csv`
- Group strip-plot figure → `<glmsingle_dir>/qc_group_strip_plots.png`

**Sections**

| # | Content |
|---|---------|
| 1 | Design matrix inspection — onset counts per stimulus per run |
| 2 | R² model comparison — types A → D, plus type-D brain map |
| 3 | FRACvalue — ridge regularization map and distribution |
| 4 | Beta quality: voxel level — per-voxel STD and mean |
| 5 | Beta quality: trial level — per-trial mean \|beta\|, outlier flagging |
| 6 | Trial metadata checks — category/stimulus completeness, NaN check |
| 7 | Group-level summary — strip plots, flag anomalous subjects |

**Note on R² interpretation:** only type-D R² is meaningful for single-trial designs (A/B/C will be near-ceiling and uninformative). They are included here for comparison only.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import image, plotting
from scipy import stats

# ---------------------------------------------------------------------------
# Paths — adjust if needed
# ---------------------------------------------------------------------------
GLMSINGLE_DIR    = Path("/home/hfluhr/shares-hare/ds-learning-habits/derivatives/glmsingle")
FMRIPREP_DIR     = Path("/home/hfluhr/shares-hare/ds-learning-habits/derivatives/fmriprep-24.0.1-noSDC")
PARTICIPANTS_TSV = Path("/home/hfluhr/data/learninghabits/participants_mvpa.tsv")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
STIM_NAMES = [
    'face_female', 'face_male',
    'figure_circle', 'figure_triangle',
    'hand_back', 'hand_palm',
    'house_1', 'house_2',
]
STIM_CATS  = ['face', 'figure', 'hand', 'house']
RUNS       = ['learning1', 'learning2', 'test']
OUTLIER_SD = 5
FRAC_MIN   = 0.05   # GLMsingle minimum FRACvalue (maximum regularization)

In [ ]:
subjects = PARTICIPANTS_TSV.read_text().strip().splitlines()
print(f"{len(subjects)} subjects")

# Check which subjects have complete outputs
MODEL_TYPED = "TYPED_FITHRF_GLMDENOISE_RR.npy"
complete, incomplete = [], []
for sub in subjects:
    sub_dir = GLMSINGLE_DIR / f"sub-{sub}"
    betas   = sub_dir / f"sub-{sub}_glmSingle_betas_CUES.nii.gz"
    info    = sub_dir / f"sub-{sub}_glmSingle_betas_CUES_info.csv"
    typed   = sub_dir / MODEL_TYPED
    if typed.exists() and betas.exists() and info.exists():
        complete.append(sub)
    else:
        incomplete.append(sub)
        missing = [f for f, p in [("typed", typed), ("betas", betas), ("info", info)] if not p.exists()]
        print(f"sub-{sub}: INCOMPLETE — missing {missing}")

print(f"\n{len(complete)}/{len(subjects)} subjects complete")

In [ ]:
# Inspect all dict keys for the first complete subject — confirm R2 and FRACvalue key names
sub0_dir = GLMSINGLE_DIR / f"sub-{complete[0]}"
for fname in ["TYPEA_ONOFF.npy", "TYPEB_FITHRF.npy",
              "TYPEC_FITHRF_GLMDENOISE.npy", "TYPED_FITHRF_GLMDENOISE_RR.npy",
              "DESIGNINFO.npy"]:
    d = np.load(sub0_dir / fname, allow_pickle=True).item()
    print(f"\n{fname}:")
    for k, v in d.items():
        shape = getattr(v, 'shape', None)
        dtype = getattr(v, 'dtype', type(v).__name__)
        print(f"  {k:40s}  shape={shape}  dtype={dtype}")

In [ ]:
# Update these after verifying key names above
R2_KEY   = 'R2'
FRAC_KEY = 'FRACvalue'

def get_brain_mask(subject):
    candidates = sorted(
        (FMRIPREP_DIR / f"sub-{subject}" / "func").glob(
            f"sub-{subject}_*_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz"
        )
    )
    if not candidates:
        raise FileNotFoundError(f"No brain mask for sub-{subject}")
    return nib.load(candidates[0])


def ensure_3d(arr, ref_shape):
    """Reshape a GLMsingle output to 3D if it arrived flattened."""
    if arr.ndim == 1:
        return arr.reshape(ref_shape)
    if arr.ndim == 4 and arr.shape[0] == 1:
        return arr[0]
    return arr

## Section 1 — Design matrix inspection

Onset counts per stimulus per run, reconstructed from `_info.csv`.  
Flags any run × condition cell with zero onsets.

In [ ]:
sec1_rows = []
for sub in subjects:
    sub_dir   = GLMSINGLE_DIR / f"sub-{sub}"
    info_path = sub_dir / f"sub-{sub}_glmSingle_betas_CUES_info.csv"
    fig_dir   = sub_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    if not info_path.exists():
        print(f"sub-{sub}: missing info CSV — skipping section 1")
        sec1_rows.append({'subject': sub, 'total_trials': np.nan, 'n_zero_onset_cells': np.nan})
        continue

    info   = pd.read_csv(info_path, index_col='trial_id')
    counts = info.pivot_table(index='stim_name', columns='run', aggfunc='size', fill_value=0)
    for r in RUNS:
        if r not in counts.columns:
            counts[r] = 0
    counts = counts.reindex(STIM_NAMES, fill_value=0)[RUNS]

    zero_cells = int((counts == 0).values.sum())

    x = np.arange(len(STIM_NAMES))
    w = 0.25
    colors = ['#5B9BD5', '#ED7D31', '#70AD47']
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for i, (run, color) in enumerate(zip(RUNS, colors)):
        ax.bar(x + (i - 1) * w, counts[run].values, w, label=run, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(STIM_NAMES, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Trial count')
    ax.set_title(f"sub-{sub} — onsets per stimulus per run"
                 + (f"  ⚠ {zero_cells} zero cells" if zero_cells else ""))
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fig.savefig(fig_dir / "qc_section1_design.png", dpi=100, bbox_inches='tight')
    plt.show(); plt.close(fig)

    if zero_cells:
        print(f"sub-{sub}: WARNING — {zero_cells} zero-onset cells")

    sec1_rows.append({'subject': sub, 'total_trials': len(info), 'n_zero_onset_cells': zero_cells})

sec1_df = pd.DataFrame(sec1_rows)
flagged = sec1_df[sec1_df['n_zero_onset_cells'] > 0]
print(f"\n{len(flagged)} subjects with zero-onset cells:")
display(flagged)

## Section 2 — R² model comparison (types A → D)

Mean R² across brain voxels per model type (bar chart) and a whole-brain type-D R² map.

**Reminder:** only type-D R² is interpretable for single-trial designs.

In [ ]:
MODEL_FILES = {
    'A: ONOFF':       'TYPEA_ONOFF.npy',
    'B: FitHRF':      'TYPEB_FITHRF.npy',
    'C: +Denoise':    'TYPEC_FITHRF_GLMDENOISE.npy',
    'D: +Ridge':      'TYPED_FITHRF_GLMDENOISE_RR.npy',
}
BAR_COLORS = ['#ADB9CA', '#9DC3E6', '#5B9BD5', '#2E75B6']

sec2_rows = []
for sub in subjects:
    sub_dir = GLMSINGLE_DIR / f"sub-{sub}"
    fig_dir = sub_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    try:
        mask_img = get_brain_mask(sub)
    except FileNotFoundError as e:
        print(e); sec2_rows.append({'subject': sub}); continue

    mask_arr  = mask_img.get_fdata(dtype=np.float32) > 0
    vol_shape = mask_arr.shape
    mean_r2s  = {}
    typed_r2  = None

    for label, fname in MODEL_FILES.items():
        fpath = sub_dir / fname
        if not fpath.exists():
            mean_r2s[label] = np.nan; continue
        d = np.load(fpath, allow_pickle=True).item()
        if R2_KEY not in d:
            print(f"sub-{sub} {fname}: key '{R2_KEY}' not found — available: {list(d.keys())}")
            mean_r2s[label] = np.nan; continue
        r2 = ensure_3d(d[R2_KEY], vol_shape)
        mean_r2s[label] = float(np.nanmean(r2[mask_arr]))
        if label.startswith('D'):
            typed_r2 = r2.copy()

    # Bar chart
    fig, ax = plt.subplots(figsize=(6, 3.5))
    labels = list(mean_r2s.keys())
    values = [mean_r2s[l] for l in labels]
    ax.bar(range(len(labels)), values, color=BAR_COLORS, alpha=0.9)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Mean R² across brain voxels (%)')
    ax.set_title(f"sub-{sub} — R² per model type")
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fig.savefig(fig_dir / "qc_section2_r2_bar.png", dpi=100, bbox_inches='tight')
    plt.show(); plt.close(fig)

    # Type-D brain map
    if typed_r2 is not None:
        typed_nib = image.new_img_like(mask_img, typed_r2)
        display   = plotting.plot_stat_map(
            typed_nib, bg_img=None, threshold=0,
            display_mode='z', cut_coords=8,
            title=f"sub-{sub} — type-D R²",
            colorbar=True, vmax=80, cmap='hot',
        )
        display.savefig(fig_dir / "qc_section2_r2_typed_map.png", dpi=100)
        display.close()

    row = {'subject': sub}
    row.update({f'mean_r2_{l[0]}': v for l, v in mean_r2s.items()})
    sec2_rows.append(row)

sec2_df = pd.DataFrame(sec2_rows)
print(sec2_df[[c for c in sec2_df.columns if 'r2' in c]].describe().round(2))

## Section 3 — FRACvalue

Ridge regularization amount per voxel.  
- `FRACvalue = 1` → no regularization needed (good SNR)  
- `FRACvalue = 0.05` → maximum regularization (minimum value, poor SNR)

Proportion of voxels at the minimum is a useful summary scalar per subject.

In [ ]:
sec3_rows = []
for sub in subjects:
    sub_dir    = GLMSINGLE_DIR / f"sub-{sub}"
    typed_path = sub_dir / MODEL_TYPED
    fig_dir    = sub_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    if not typed_path.exists():
        sec3_rows.append({'subject': sub, 'frac_min_prop': np.nan, 'frac_mean': np.nan})
        continue

    try:
        mask_img = get_brain_mask(sub)
    except FileNotFoundError as e:
        print(e); sec3_rows.append({'subject': sub, 'frac_min_prop': np.nan, 'frac_mean': np.nan}); continue

    mask_arr  = mask_img.get_fdata(dtype=np.float32) > 0
    vol_shape = mask_arr.shape
    typed     = np.load(typed_path, allow_pickle=True).item()

    if FRAC_KEY not in typed:
        print(f"sub-{sub}: '{FRAC_KEY}' not found — available: {list(typed.keys())}")
        sec3_rows.append({'subject': sub, 'frac_min_prop': np.nan, 'frac_mean': np.nan}); continue

    frac = ensure_3d(typed[FRAC_KEY], vol_shape)
    frac_masked   = frac[mask_arr]
    frac_min_prop = float((frac_masked <= FRAC_MIN + 1e-6).mean())
    frac_mean     = float(np.nanmean(frac_masked))

    # Brain map
    frac_nib = image.new_img_like(mask_img, frac)
    d = plotting.plot_stat_map(
        frac_nib, bg_img=None, threshold=0,
        display_mode='z', cut_coords=8,
        title=f"sub-{sub} — FRACvalue (min={FRAC_MIN}=max reg, 1=no reg)",
        colorbar=True, vmin=0, vmax=1, cmap='RdYlGn',
    )
    d.savefig(fig_dir / "qc_section3_frac_map.png", dpi=100)
    d.close()

    # Histogram
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(frac_masked, bins=50, color='steelblue', alpha=0.85, edgecolor='white')
    ax.axvline(FRAC_MIN, color='red', linestyle='--', linewidth=1.2,
               label=f'Min={FRAC_MIN} ({frac_min_prop*100:.1f}% of voxels)')
    ax.set_xlabel('FRACvalue'); ax.set_ylabel('Voxels')
    ax.set_title(f"sub-{sub} — FRACvalue distribution")
    ax.legend(fontsize=9); ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fig.savefig(fig_dir / "qc_section3_frac_hist.png", dpi=100)
    plt.show(); plt.close(fig)

    sec3_rows.append({'subject': sub, 'frac_min_prop': frac_min_prop, 'frac_mean': frac_mean})

sec3_df = pd.DataFrame(sec3_rows)
print(sec3_df[['frac_min_prop', 'frac_mean']].describe().round(3))

## Sections 4 & 5 — Beta quality: voxel level and trial level

Loads the 4D beta NIfTI once per subject, applies the brain mask, then:

**Section 4 — voxel level**
- Per-voxel STD and mean beta across trials
- Brain maps + STD distribution (right tail = unstable voxels)

**Section 5 — trial level**
- Per-trial mean |beta| across masked voxels
- Time series colour-coded by run; outliers flagged at ±`OUTLIER_SD` SD
- Distribution of per-trial mean betas

In [ ]:
RUN_COLORS = {'learning1': '#5B9BD5', 'learning2': '#ED7D31', 'test': '#70AD47'}

sec45_rows = []
for sub in subjects:
    sub_dir    = GLMSINGLE_DIR / f"sub-{sub}"
    betas_path = sub_dir / f"sub-{sub}_glmSingle_betas_CUES.nii.gz"
    info_path  = sub_dir / f"sub-{sub}_glmSingle_betas_CUES_info.csv"
    fig_dir    = sub_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    if not betas_path.exists():
        print(f"sub-{sub}: betas NIfTI missing — skipping sections 4 & 5")
        sec45_rows.append({'subject': sub, 'mean_voxel_std': np.nan, 'n_nan_voxels': np.nan,
                           'n_outlier_trials': np.nan, 'n_nan_trials': np.nan})
        continue

    try:
        mask_img = get_brain_mask(sub)
    except FileNotFoundError as e:
        print(e); sec45_rows.append({'subject': sub, 'mean_voxel_std': np.nan, 'n_nan_voxels': np.nan,
                                     'n_outlier_trials': np.nan, 'n_nan_trials': np.nan}); continue

    mask_arr  = mask_img.get_fdata(dtype=np.float32) > 0
    betas_4d  = nib.load(betas_path).get_fdata(dtype=np.float32)  # (x, y, z, n_trials)
    betas_2d  = betas_4d[mask_arr, :]                              # (n_voxels, n_trials)

    info = pd.read_csv(info_path, index_col='trial_id') if info_path.exists() else None

    # -----------------------------------------------------------------------
    # Section 4: voxel level
    # -----------------------------------------------------------------------
    voxel_std  = betas_2d.std(axis=1)
    voxel_mean = betas_2d.mean(axis=1)
    n_nan_vox  = int(np.isnan(betas_2d).any(axis=1).sum())

    def _make_vol(vals):
        vol = np.zeros(mask_arr.shape, dtype=np.float32)
        vol[mask_arr] = vals
        return image.new_img_like(mask_img, vol)

    for arr, tag, title in [
        (voxel_std,  'std',  'per-voxel beta STD'),
        (voxel_mean, 'mean', 'per-voxel mean beta'),
    ]:
        d = plotting.plot_stat_map(
            _make_vol(arr), bg_img=None, threshold=0.01,
            display_mode='z', cut_coords=8,
            title=f"sub-{sub} — {title}",
            colorbar=True,
        )
        d.savefig(fig_dir / f"qc_section4_voxel_{tag}_map.png", dpi=100)
        d.close()

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(voxel_std[np.isfinite(voxel_std)], bins=60, color='steelblue', alpha=0.85, edgecolor='white')
    ax.set_xlabel('Per-voxel beta STD'); ax.set_ylabel('Voxels')
    ax.set_title(f"sub-{sub} — voxel STD  |  {n_nan_vox} NaN voxels")
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fig.savefig(fig_dir / "qc_section4_voxel_std_hist.png", dpi=100)
    plt.show(); plt.close(fig)

    # -----------------------------------------------------------------------
    # Section 5: trial level
    # -----------------------------------------------------------------------
    trial_mean_abs = np.nanmean(np.abs(betas_2d), axis=0)  # (n_trials,)
    n_nan_trials   = int(np.isnan(betas_2d).any(axis=0).sum())
    sub_mean       = np.nanmean(trial_mean_abs)
    sub_std        = np.nanstd(trial_mean_abs)
    outlier_thresh = sub_mean + OUTLIER_SD * sub_std
    outlier_mask   = trial_mean_abs > outlier_thresh
    n_outliers     = int(outlier_mask.sum())

    # Time series
    fig, ax = plt.subplots(figsize=(14, 3.5))
    if info is not None:
        for run, color in RUN_COLORS.items():
            idx = [i for i in range(len(trial_mean_abs))
                   if i < len(info) and info.iloc[i]['run'] == run]
            if idx:
                ax.scatter(idx, trial_mean_abs[idx], s=10, color=color, label=run, alpha=0.7, zorder=3)
    else:
        ax.plot(trial_mean_abs, color='steelblue', linewidth=0.8)
    ax.axhline(outlier_thresh, color='red', linestyle='--', linewidth=1.2,
               label=f'{OUTLIER_SD} SD threshold ({n_outliers} outliers)')
    ax.set_xlabel('Trial index'); ax.set_ylabel('Mean |beta| across voxels')
    ax.set_title(f"sub-{sub} — trial-level mean |beta|  |  {n_nan_trials} NaN trials")
    ax.legend(fontsize=8); ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fig.savefig(fig_dir / "qc_section5_trial_timeseries.png", dpi=100)
    plt.show(); plt.close(fig)

    # Distribution
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(trial_mean_abs[np.isfinite(trial_mean_abs)], bins=30,
            color='steelblue', alpha=0.85, edgecolor='white')
    ax.axvline(outlier_thresh, color='red', linestyle='--', linewidth=1.2,
               label=f'{OUTLIER_SD} SD ({n_outliers} outliers)')
    ax.set_xlabel('Mean |beta| across voxels'); ax.set_ylabel('Trials')
    ax.set_title(f"sub-{sub} — trial mean |beta| distribution")
    ax.legend(fontsize=9); ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fig.savefig(fig_dir / "qc_section5_trial_dist.png", dpi=100)
    plt.show(); plt.close(fig)

    sec45_rows.append({
        'subject':          sub,
        'mean_voxel_std':   float(np.nanmean(voxel_std)),
        'n_nan_voxels':     n_nan_vox,
        'n_outlier_trials': n_outliers,
        'n_nan_trials':     n_nan_trials,
    })

sec45_df = pd.DataFrame(sec45_rows)
print(sec45_df.describe().round(3))

## Section 6 — Trial metadata checks

From `_info.csv`: trial counts per stimulus category per run, completeness check (all 4 categories and all 8 stimuli present in each run), and NaN check in the betas volume.

In [ ]:
sec6_rows = []
for sub in subjects:
    sub_dir    = GLMSINGLE_DIR / f"sub-{sub}"
    info_path  = sub_dir / f"sub-{sub}_glmSingle_betas_CUES_info.csv"
    betas_path = sub_dir / f"sub-{sub}_glmSingle_betas_CUES.nii.gz"

    if not info_path.exists():
        sec6_rows.append({'subject': sub, 'n_missing_run_cat_combos': np.nan,
                          'n_missing_run_stim_combos': np.nan, 'n_nan_trials_volume': np.nan})
        continue

    info = pd.read_csv(info_path, index_col='trial_id')

    # Category counts per run
    cat_counts = info.groupby(['run', 'stim_cat']).size().unstack(fill_value=0)

    # Completeness checks
    exp_cats  = set(STIM_CATS)
    exp_stims = set(STIM_NAMES)
    missing_cat_combos  = 0
    missing_stim_combos = 0
    for run in RUNS:
        run_df = info[info['run'] == run]
        missing_cats  = exp_cats  - set(run_df['stim_cat'].unique())
        missing_stims = exp_stims - set(run_df['stim_name'].unique())
        if missing_cats or missing_stims:
            print(f"sub-{sub}  {run}: missing cats={missing_cats}  stims={missing_stims}")
        missing_cat_combos  += len(missing_cats)
        missing_stim_combos += len(missing_stims)

    # NaN check in betas volume
    if betas_path.exists():
        betas_4d   = nib.load(betas_path).get_fdata(dtype=np.float32)
        nan_trials = int(np.isnan(betas_4d).any(axis=(0, 1, 2)).sum())
    else:
        nan_trials = np.nan

    if missing_cat_combos == 0 and missing_stim_combos == 0 and nan_trials == 0:
        pass  # clean subject, no need to print
    else:
        print(f"sub-{sub}: {missing_cat_combos} missing cat/run combos, "
              f"{missing_stim_combos} missing stim/run combos, {nan_trials} NaN trial betas")

    sec6_rows.append({
        'subject':                  sub,
        'n_missing_run_cat_combos': missing_cat_combos,
        'n_missing_run_stim_combos': missing_stim_combos,
        'n_nan_trials_volume':      nan_trials,
    })

sec6_df = pd.DataFrame(sec6_rows)
issues = sec6_df[(sec6_df['n_missing_run_cat_combos'] > 0) |
                 (sec6_df['n_missing_run_stim_combos'] > 0) |
                 (sec6_df['n_nan_trials_volume'] > 0)]
print(f"\n{len(issues)} subjects with issues:")
display(issues)

## Section 7 — Group-level summary

Aggregates scalar QC metrics across all subjects. Strip plots flag anomalous subjects (> 3 SD from group mean).

In [ ]:
# Merge all per-subject metrics
group_df = sec1_df.copy()
for df in [sec2_df, sec3_df, sec45_df, sec6_df]:
    group_df = group_df.merge(df, on='subject', how='outer')

group_df.to_csv(GLMSINGLE_DIR / "qc_group_summary.csv", index=False)
print(f"Saved group summary: {len(group_df)} subjects")
display(group_df.describe().round(3))

In [ ]:
STRIP_METRICS = {
    'mean_r2_D':        'Mean type-D R² (brain, %)',
    'frac_min_prop':    'Fraction voxels at min FRACvalue',
    'mean_voxel_std':   'Mean per-voxel beta STD',
    'n_outlier_trials': 'Outlier trials (count)',
    'n_nan_voxels':     'NaN voxels (count)',
}

available = {k: v for k, v in STRIP_METRICS.items() if k in group_df.columns}
n = len(available)
fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4.5))
if n == 1:
    axes = [axes]

rng = np.random.default_rng(42)
for ax, (col, label) in zip(axes, available.items()):
    vals = group_df[col].values.astype(float)
    finite = vals[np.isfinite(vals)]
    jitter = rng.uniform(-0.15, 0.15, len(finite))
    subs   = group_df.loc[np.isfinite(vals), 'subject'].values

    ax.scatter(jitter, finite, s=20, alpha=0.7, color='steelblue',
               edgecolors='white', linewidth=0.4, zorder=3)
    ax.axhline(np.mean(finite), color='black', linewidth=1.5,
               label=f'Mean = {np.mean(finite):.3f}', zorder=4)

    # Flag > 3 SD outliers
    thresh_hi = np.mean(finite) + 3 * np.std(finite)
    thresh_lo = np.mean(finite) - 3 * np.std(finite)
    for sub, jit, val in zip(subs, jitter, finite):
        if val > thresh_hi or val < thresh_lo:
            ax.annotate(f"sub-{sub}", (jit, val), fontsize=6,
                        ha='center', va='bottom', color='red')

    ax.set_xticks([])
    ax.set_ylabel(label, fontsize=9)
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
    ax.spines[['top', 'right', 'bottom']].set_visible(False)

plt.suptitle("Group QC metrics — each dot = one subject  |  red labels = >3 SD outliers", y=1.02)
plt.tight_layout()
fig.savefig(GLMSINGLE_DIR / "qc_group_strip_plots.png", dpi=100, bbox_inches='tight')
plt.show()